# 라이브러리 및 데이터 전처리 부분

In [2]:
# 라이브러리 Import
import pandas as pd
import seaborn as sns
import folium
import json
import plotly.express as px
import matplotlib.pyplot as plt
from folium import CircleMarker
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

In [3]:
# CSV 파일 로드
df = pd.read_csv('데이터/통합/통합_데이터.csv')

In [4]:
# 신규 개업역의 경우에는 개업일 이후에만 분석하기 위해서 실제 있는 신규 개업역에 대해서 딕셔너리 및 딕셔너리 이용 필터링 함수 선언

# 신규 개업 역 (역명: 개업일)
new_stations = {
    "관악산(서울대)": "2022-05-28",
    "당곡": "2022-05-28",
    "보라매공원": "2022-05-28",
    "보라매병원": "2022-05-28",
    "서울대벤처타운": "2022-05-28",
    "서울지방병무청": "2022-05-28",
    "서원": "2022-05-28",
    "성남": "2024-03-30",
    "암사역사공원": "2024-08-10",
    "원종": "2023-07-01"
}

# 신규 개업역의 개업일 이후 데이터만 필터링
def filter_new_station_data(df, new_stations):
    df_filtered = df.copy()
    for station, open_date in new_stations.items():
        open_date = pd.to_datetime(open_date)
        df_filtered = df_filtered[~((df_filtered['역명'] == station) & (df_filtered['일시'] < open_date))]
    return df_filtered

In [5]:
# 일시를 datetime으로 변환
df['일시'] = pd.to_datetime(df['일시'])

# 필터링 데이터 생성
df_filtered = filter_new_station_data(df, new_stations)

# CSV 데이터 정리 부분

In [6]:
# CSV 데이터 정리

for col in df.columns:
    print(f"열 이름: {col}")
    print(f" - 데이터 타입: {df[col].dtype}")
    print(f" - 결측값 수: {df[col].isnull().sum()}")
    print(f" - 고유값 수: {df[col].nunique()}")
    print(f" - 고유값 예시: {df[col].unique()[:10]}")
    print("-" * 50)

열 이름: 일시
 - 데이터 타입: datetime64[ns]
 - 결측값 수: 0
 - 고유값 수: 1096
 - 고유값 예시: <DatetimeArray>
['2022-01-01 00:00:00', '2022-01-02 00:00:00', '2022-01-03 00:00:00',
 '2022-01-04 00:00:00', '2022-01-05 00:00:00', '2022-01-06 00:00:00',
 '2022-01-07 00:00:00', '2022-01-08 00:00:00', '2022-01-09 00:00:00',
 '2022-01-10 00:00:00']
Length: 10, dtype: datetime64[ns]
--------------------------------------------------
열 이름: 역명
 - 데이터 타입: object
 - 결측값 수: 0
 - 고유값 수: 521
 - 고유값 예시: ['4.19민주묘지' '가능' '가락시장' '가산디지털단지' '가양' '가오리' '가좌' '가천대' '가평' '간석']
--------------------------------------------------
열 이름: 위도
 - 데이터 타입: float64
 - 결측값 수: 0
 - 고유값 수: 521
 - 고유값 예시: [37.649502 37.748577 37.492245 37.480338 37.561391 37.641537 37.568491
 37.448605 37.814536 37.464737]
--------------------------------------------------
열 이름: 경도
 - 데이터 타입: float64
 - 결측값 수: 0
 - 고유값 수: 521
 - 고유값 예시: [127.013684 127.044213 127.117757 126.882656 126.854456 127.016789
 126.915487 127.126697 127.510739 126.694181]
-------------

# 실제 데이터 분석 부분

## 계절 - 날씨 유형별 상대 증가율 Top 10에 대한 리스트 생성 및 출력

In [7]:
# 계절-날씨 유형 그룹별로 각 역의 유입량 상대 증가율을 계산하고,
# 주말/공휴일 기준에서 각 그룹의 증가율 상위 10개 역 리스트를 생성 및 출력하는 코드 (계절 평균 기준)

# ✅ 0. 주말/공휴일 기준 필터링
df_holiday = df_filtered[df_filtered['휴일 구분'] == '주말/공휴일']

# ✅ 1. 계절-역별 평균 유입량 계산 (기준: 해당 계절 전체)
season_avg = df_holiday.groupby(['계절', '역명'])['인구유입량'].mean().reset_index()
season_avg = season_avg.rename(columns={'인구유입량': '계절평균'})

# ✅ 2. 주말/공휴일 기준 조건 평균 유입량 계산 (계절-날씨-역)
grouped = df_holiday.groupby(['계절', '날씨 유형', '역명'])['인구유입량'].mean().reset_index()
grouped = grouped.rename(columns={'인구유입량': '조건평균'})

# ✅ 3. 기준 계절 평균과 병합
grouped = grouped.merge(season_avg, on=['계절', '역명'])

# ✅ 4. 상대 증가율 계산 (조건평균 / 계절평균 - 1)
grouped['상대증가율'] = grouped['조건평균'] / grouped['계절평균'] - 1

# ✅ 5. 계절-날씨 그룹별 Top 10 저장 및 출력
group_top10_dict = {}

for (season, weather), group in grouped.groupby(['계절', '날씨 유형']):
    top10 = group.sort_values('상대증가율', ascending=False).head(10).copy()
    top10['계절평균'] = top10['계절평균'].round(0).astype(int)
    top10['조건평균'] = top10['조건평균'].round(0).astype(int)
    top10['상대증가율(%)'] = (top10['상대증가율'] * 100).round(1)

    # 정리된 컬럼 순서
    top10 = top10[['역명', '계절평균', '조건평균', '상대증가율(%)']].reset_index(drop=True)
    group_top10_dict[(season, weather)] = top10

    # 출력
    print(f"\n▶ {season} - {weather} 조건 (휴일 기준, 계절 평균 기준) 상위 10 역")
    print(top10.to_string(index=True))


▶ 가을 - 강우 조건 (휴일 기준, 계절 평균 기준) 상위 10 역
         역명   계절평균   조건평균  상대증가율(%)
0        성남    690    846      22.7
1       학여울   2081   2181       4.8
2      소래포구   6616   6932       4.8
3    암사역사공원   3971   3941      -0.8
4        도원   2668   2644      -0.9
5       봉은사  14627  14418      -1.4
6  삼성(무역센터)  30019  29470      -1.8
7      김포공항  25271  24721      -2.2
8        월곶   2250   2196      -2.4
9        안산  11149  10862      -2.6

▶ 가을 - 대설 조건 (휴일 기준, 계절 평균 기준) 상위 10 역
         역명   계절평균   조건평균  상대증가율(%)
0        구일   6598  18923     186.8
1    공항화물청사   1363   2253      65.3
2        화서   7416  11208      51.1
3        성수  31794  46460      46.1
4       학여울   2081   2948      41.7
5      둔촌오륜    839   1152      37.3
6  인천공항2터미널   5180   6974      34.6
7        구성   3782   4875      28.9
8        탕정   1178   1502      27.5
9        덕계   3308   4204      27.1

▶ 가을 - 맑음 조건 (휴일 기준, 계절 평균 기준) 상위 10 역
           역명   계절평균   조건평균  상대증가율(%)
0         한양대   6017   6869      14.2
1         대공

## 계절 - 날씨 유형별 상대 증가율 Top 10 리스트 통해서 지도에 시각화

In [8]:
# 계절 - 날씨 유형별 상대 증가율 Top 10 리스트를 통해서 지도에 시각화하는 코드 (계절 평균 기준, folium 이용)

# 바운더리 GeoJSON 경로
geojson_path = '데이터/JSON/seoul_neighborhoods_geo_utf8.json'
with open(geojson_path, 'r', encoding='utf-8') as f:
    geojson_data = json.load(f)

# 날씨 유형별 색상 정의
weather_color_map = {
    '맑음': 'gold',
    '흐림': 'gray',
    '보통': 'lightblue',
    '강우': 'blue',
    '약우': 'lightskyblue',
    '대설': 'darkred',
    '적설': 'purple',
    '고습': 'green'
}

# ✅ 주말/공휴일만 필터링
df_holiday = df_filtered[df_filtered['휴일 구분'] == '주말/공휴일']

# 계절별 지도 생성
for season in ['봄', '여름', '가을', '겨울']:
    season_data = []

    for weather in df_holiday['날씨 유형'].unique():
        key = (season, weather)
        if key not in group_top10_dict:
            continue
        
        df_top = group_top10_dict[key].copy()
        df_top['계절'] = season
        df_top['날씨 유형'] = weather
        
        # 상대 증가율 수치화 및 필터링
        df_top['상대증가율'] = df_top['상대증가율(%)'].astype(str).str.replace('%', '').astype(float)
        df_top = df_top[df_top['상대증가율'] >= 10.0]

        # 위경도 병합
        coords = df_holiday[['역명', '위도', '경도']].drop_duplicates()
        df_top = df_top.merge(coords, on='역명', how='left')

        season_data.append(df_top)

    if not season_data:
        continue

    df_map = pd.concat(season_data, ignore_index=True)

    # 지도 초기화
    fmap = folium.Map(location=[37.5665, 126.9780], zoom_start=11, tiles='CartoDB positron')

    # 바운더리 추가
    folium.Choropleth(
        geo_data=geojson_data,
        fill_color='lightgray',
        fill_opacity=0.2,
        line_color='black',
        line_weight=1.5,
        highlight=True,
        name='서울시 경계선'
    ).add_to(fmap)

    # 마커 추가
    for _, row in df_map.iterrows():
        folium.CircleMarker(
            location=(row['위도'], row['경도']),
            radius=3 + row['상대증가율'] * 0.15,
            color=weather_color_map.get(row['날씨 유형'], 'gray'),
            fill=True,
            fill_opacity=0.8,
            popup=folium.Popup(
                f"<b>{row['역명']}</b><br>날씨: {row['날씨 유형']}<br>증가율: {row['상대증가율']}%", max_width=250
            )
        ).add_to(fmap)

    fmap.save(f"{season}_휴일_날씨_상대증가율_지도.html")